# Practice 2.5 · 2023 vs 2026 对照

> **对应知识点**：EP2 · KP5 · 2026 视角综合
> **测什么**：让你**亲眼看到** 2023 教科书上的老招在 4.5+ 上没用甚至反效果
> **预计时间**：8 分钟

---

## 你会看到什么

同一任务两种 prompt：
- 版本 A · 2023 老招堆砌（`【非常重要！！！】` / `世界顶级` / `让我们一步一步思考` / 三引号 delimiter）
- 版本 B · 2026 标准（简洁 + 上限暗示词）

**你会发现：老招不但没帮上，反而让输出变啰嗦 / 更"AI 腔"**。

这是"教科书没告诉你的" 时刻——教你**别再抄老 prompt 模板**。


In [ ]:
# ============ 前置：安装 SDK + 配置 client ============
!pip install openai -q

from openai import OpenAI
from google.colab import userdata   # Colab；Modelscope 上用 os.environ 或直接填 key

# 推荐用 Secrets 存 key（不要硬编码）
API_KEY = userdata.get("DEEPSEEK_KEY")   # Colab Secrets
# 或 Modelscope 上：
# API_KEY = "sk-你的DeepSeek key"

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.deepseek.com/v1"
)

MODEL = "deepseek-chat"

def call(prompt, **kwargs):
    """简化的 chat 调用。默认 temperature=0 保证实验可重现。"""
    kwargs.setdefault("temperature", 0)
    kwargs.setdefault("max_tokens", 512)
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        **kwargs
    )
    return r.choices[0].message.content


## 相同任务 · 两种 Prompt

In [ ]:
# 任务：重构一段代码

CODE = """
def f(x):
    return x*2 if x>0 else -x*2
"""

# 版本 A · 2023 老招堆砌
PROMPT_A = f"""【非常重要！！！】你是世界顶级的、资深的 Python 专家！
让我们一步一步思考。请你务必、一定、无论如何要遵循最佳实践，
把下面这段代码"""重构成更清晰的版本"""：

{CODE}

请深入分析每一个细节，做得越详细越好！！！"""

# 版本 B · 2026 标准
PROMPT_B = f"""把下面代码重构为更清晰的版本。

<code>
{CODE}
</code>

要求：
- 用有语义的函数名和变量名
- 加简短 docstring 说明输入输出
- 保持功能不变
- 尽可能考虑边界情况(负数、0、超大数、非数字输入)"""

## 跑测试 · 各跑 3 次

In [ ]:
print("=== 版本 A · 2023 老招堆砌 ===")
for i in range(3):
    print(f"\n--- A 第 {i+1} 次 ---")
    out = call(PROMPT_A, max_tokens=1500)
    print(f"长度：{len(out)} 字")
    print(out[:300] + "...")

print("\n\n=== 版本 B · 2026 标准 ===")
for i in range(3):
    print(f"\n--- B 第 {i+1} 次 ---")
    out = call(PROMPT_B, max_tokens=1500)
    print(f"长度：{len(out)} 字")
    print(out[:300] + "...")

## 观察点 · 主观打分

跑完后自己判断（不用严格打分，直觉即可）：

| 维度 | A 老招 | B 2026 |
|---|---|---|
| **代码质量**（重构得怎么样） | ? | ? |
| **输出啰嗦度**（有没有 AI 腔） | ? | ? |
| **是否加"作为世界顶级专家我认为..."之类的表演话** | ? | ? |
| **实际有用度** | ? | ? |

**多数情况下你会发现 B 更实用**——老招不但没帮忙，反而让输出加了一堆自我表演的话。

## 为什么老招塌了

Claude 4.5+ 起：
- 对 `CRITICAL!` / 多感叹号 **overtrigger** → 输出过度防御、过度啰嗦
- 对 `world-class expert` 夸张修饰 → 加"作为专家我认为..."的表演腔
- 对 `Let's think step by step` → 4.5+ 默认就在 reason，这句是 noise
- 对 `"""三引号"""` → 早就被 XML tag 替代

## 一句心法

> **2023 的 prompt 教科书 90% 是错的**。
>
> 你从旧文章 / 旧教程学的招，在 Claude 4.5+ / GPT-5 / DeepSeek V4 上大多是 noise。
> 真正管用的是：**清晰 + 具体 + 上限暗示词** 三招——**外加一堆 2023 教程没教的东西**（XML tags / grounded by quoting / Structured Outputs）。
